# Data 304 — NLP Wrangling Demo

## Environment setup
Run this once if needed:
```bash
pip install spacy pandas matplotlib
python -m spacy download en_core_web_sm
```


In [ ]:
# Imports and model load
import spacy
from spacy import displacy
from spacy.matcher import Matcher
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise OSError(
        "Model 'en_core_web_sm' is not installed. Run: python -m spacy download en_core_web_sm"
    )

TEXT = "UT Knoxville launched a new Data Science program in Fall 2025. Python processes data efficiently. Efficient data processing methods improve performance."
doc = nlp(TEXT)
TEXT

## Chunking and Phrases
spaCy exposes *noun chunks* directly.

In [ ]:
list(chunk.text for chunk in doc.noun_chunks)

### Simple verb phrase patterns with `Matcher`
This is a naive pattern: VERB optionally followed by one or more NOUNs.

In [ ]:
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "VERB"}, {"POS": "NOUN", "OP": "*"}]
matcher.add("VERB_PHRASE", [pattern])
matches = matcher(doc)
[(doc[start:end].text, start, end) for _, start, end in matches]

In [ ]:
matcher = Matcher(nlp.vocab)
pattern = [{"POS": "VERB"}, {"POS": "NOUN", "OP": "+"}]
matcher.add("VERB_PHRASE", [pattern])
matches = matcher(doc)
[(doc[start:end].text, start, end) for _, start, end in matches]

## From Tokens to Features
Build a table of linguistic annotations.

In [ ]:
rows = [(t.text, t.lemma_, t.pos_, t.dep_, t.ent_type_) for t in doc]
df = pd.DataFrame(rows, columns=["Token", "Lemma", "POS", "Dep", "Entity"]) 
df

In [ ]:
features = {
    "num_tokens": len(df),
    "num_unique_tokens": df["Token"].nunique(),
    "avg_token_length": df["Token"].str.len().mean(),
    "num_entities": (df["Entity"] != "").sum(),
    "noun_ratio": (df["POS"] == "NOUN").mean()
}

pd.DataFrame([features])


## Quantitative Entity Summary 
Count and visualize entity distribution across the text.

In [ ]:
counts = Counter(ent.label_ for ent in doc.ents)
counts

In [ ]:
if counts:
    labels, values = zip(*counts.items())
    plt.figure()
    plt.bar(labels, values)
    plt.title("Entity label counts")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.show()
else:
    print("No entities in text.")

## Custom Entity Rules
Use `EntityRuler` to add domain-specific entities (e.g., skills).

In [ ]:
from spacy.pipeline import EntityRuler
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "Pandas"}
]
ruler.add_patterns(patterns)

doc2 = nlp("Python and Pandas are popular in data science.")
[(ent.text, ent.label_) for ent in doc2.ents]